# Schneider cubic Bezier fitting laboratory
Fit a piecewise cubic Bezier chain to digitised samples. The tolerance is a maximum squared residual, not a distance in coordinate units.

In [ ]:
import sys, numpy as np
from pathlib import Path
sys.path.insert(0,str(Path.cwd().parent/'implementations/python/curves-smoothing'))
from curve_fitting import fit_bezier_curves
t=np.linspace(0,1,40)
source=np.array([[0,0],[1,2],[3,2],[4,0]],dtype=float)
def cubic(c,u):
    v=1-u
    return c[0]*v**3+3*c[1]*v*v*u+3*c[2]*v*u*u+c[3]*u**3
samples=np.array([cubic(source,u) for u in t])
samples+=np.column_stack((.015*np.sin(17*t),.015*np.cos(13*t)))
curves=fit_bezier_curves(samples, error_tolerance=.01, max_iterations=5)
len(curves)


In [ ]:
def eval_curve(c,u):
    v=1-u
    return c[0]*v**3+3*c[1]*v*v*u+3*c[2]*v*u*u+c[3]*u**3
fitted=np.vstack([np.array([eval_curve(np.asarray(c),u) for u in np.linspace(0,1,50)]) for c in curves])
import matplotlib.pyplot as plt
plt.figure(figsize=(9,5));plt.plot(samples[:,0],samples[:,1],'.',label='digitised samples');
for i,c in enumerate(curves):
    z=np.array([eval_curve(np.asarray(c),u) for u in np.linspace(0,1,80)])
    plt.plot(z[:,0],z[:,1],label=f'Bezier segment {i+1}')
plt.axis('equal');plt.legend();plt.grid(True)


## Interpretation
The fitter begins with chord-length parameterisation and least-squares tangent handles. If error is close to tolerance it tries Newton–Raphson reparameterisation; otherwise it splits at the maximum-error sample and recurses. Inspect how the tolerance changes segment count. The fit preserves endpoints and split continuity, but does not promise tangent continuity at every split.